# Notebook for the 1000 Runs Ensemble

In [111]:
import pandas as pd
import os
from utils.eda_utils import EDAUtils
import boto3

In [ ]:
%load_ext autoreload
%autoreload 2

In [112]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
MAPPING_DIR_PATH = os.path.join(DATA_DIR_PATH, "mapping")
SSP_DIR_PATH = os.path.join(DATA_DIR_PATH, "ssp")
TRAINING_DIR_PATH = os.path.join(DATA_DIR_PATH, "training")
CONFIG_DIR_PATH = os.path.join(ROOT_DIR_PATH, "config")

In [ ]:
os.makedirs(SSP_DIR_PATH, exist_ok=True)

In [113]:
edau = EDAUtils()

## Pull data from AWS S3


In [ ]:
aws_config = edau.read_yaml(os.path.join(CONFIG_DIR_PATH, "aws_credentials_config.yaml"))
profile_name = aws_config["profile_name"]
bucket_name = aws_config["bucket_name"]
# Set your profile
session = boto3.Session(profile_name=profile_name)

# Create an S3 client or resource
s3 = session.resource('s3')

run_id = "sisepuede_run_2025-08-28t15;29;22.344855"

# Define folder prefix
prefix = f'transfers/{run_id}/'  # this is like the "folder" in S3

In [ ]:
# Local destination
destination = os.path.join(SSP_DIR_PATH, run_id)
if os.path.exists(destination) and os.listdir(destination):
    print(f"Destination '{destination}' already exists and is not empty. Skipping download.")
else:
    os.makedirs(destination, exist_ok=True)
    bucket = s3.Bucket(bucket_name)
    for obj in bucket.objects.filter(Prefix=prefix):
        if obj.key.endswith('/') or "transformations" in obj.key:  # skip directories and transformations
            continue
        target_path = os.path.join(destination, os.path.basename(obj.key))
        print(f"Downloading {obj.key} to {target_path}")
        bucket.download_file(obj.key, target_path)

In [114]:
run_id="sisepuede_run_2025-09-18t09;19;22.726476"

In [115]:
SIMULATION_DIR_PATH = os.path.join(SSP_DIR_PATH, run_id)
print(SIMULATION_DIR_PATH)

e:\Current_2023\WI\work\ssp_louisiana\metamodel\data\ssp\sisepuede_run_2025-09-18t09;19;22.726476


## Load and Process LHC Samples Dataframes

In [116]:
# Load lhc samples dfs
lhs_exogenous_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES.csv"))
lhs_levers_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_LEVER_EFFECTS.csv"))

In [117]:
# Check design ids
lhs_exogenous_df.head()

,region,design_id,future_id,46,47,48,49,50,51,52,53,54,55,56,57,58
0,louisiana,-1,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,0.508737,0.781123,0.313322,0.792661,0.700647
1,louisiana,-1,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,0.973614,0.134799,0.862360,0.031865,0.682710
2,louisiana,-1,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,0.053175,0.541365,0.063797,0.578509,0.348348
3,louisiana,-1,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,0.885947,0.628554,0.487522,0.536350,0.855946
4,louisiana,-1,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,0.223495,0.396805,0.876480,0.048358,0.042901


In [118]:
lhs_exogenous_df.design_id.unique()

array([-1,  4,  0])

In [119]:
lhs_exogenous_df[lhs_exogenous_df['design_id']== 0]

,region,design_id,future_id,46,47,48,49,50,51,52,53,54,55,56,57,58
2000,louisiana,0,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,0.508737,0.781123,0.313322,0.792661,0.700647
2001,louisiana,0,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,0.973614,0.134799,0.862360,0.031865,0.682710
2002,louisiana,0,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,0.053175,0.541365,0.063797,0.578509,0.348348
2003,louisiana,0,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,0.885947,0.628554,0.487522,0.536350,0.855946
2004,louisiana,0,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,0.223495,0.396805,0.876480,0.048358,0.042901
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,louisiana,0,996,0.535777,0.643117,0.039013,0.511673,0.770811,0.214445,0.991425,0.880307,0.590308,0.274079,0.276883,0.815408,0.150082
2996,louisiana,0,997,0.776126,0.825906,0.960728,0.400776,0.776082,0.496014,0.795152,0.557662,0.782966,0.993660,0.405487,0.710320,0.501973
2997,louisiana,0,998,0.117222,0.783518,0.845782,0.771561,0.250421,0.733586,0.592633,0.047925,0.482816,0.986294,0.810759,0.957187,0.471602
2998,louisiana,0,999,0.960680,0.188435,0.975613,0.017300,0.272812,0.873458,0.358376,0.094290,0.017529,0.128812,0.389486,0.979377,0.900547


In [120]:
lhs_levers_df.head()

,region,design_id,future_id,1,2,3,4,5,6,7,...,1759,1760,1765,1766,1768,1776,1778,1790,1791,1799
0,louisiana,-1,1,0.073998,0.692822,0.979482,0.051058,0.188261,0.581816,0.698145,...,0.644272,0.214180,0.955481,0.733736,0.116532,0.828401,0.217715,0.589410,0.939430,0.804396
1,louisiana,-1,2,0.425023,0.729207,0.384489,0.436918,0.621826,0.168596,0.725935,...,0.486423,0.278023,0.680104,0.626837,0.392910,0.005622,0.342793,0.767436,0.435921,0.631640
2,louisiana,-1,3,0.881846,0.761554,0.325130,0.205597,0.237030,0.682775,0.718216,...,0.331039,0.193177,0.968159,0.467609,0.402445,0.099340,0.676226,0.109182,0.582023,0.877888
3,louisiana,-1,4,0.075106,0.542313,0.668099,0.577434,0.767079,0.804253,0.966784,...,0.267909,0.674077,0.246089,0.464519,0.968246,0.733699,0.143231,0.289890,0.794136,0.147144
4,louisiana,-1,5,0.057338,0.477879,0.866591,0.161689,0.282495,0.282664,0.310182,...,0.135390,0.111915,0.455861,0.378087,0.267354,0.528790,0.529280,0.660236,0.690165,0.539415


In [121]:
lhs_levers_df.design_id.unique()

array([-1,  4,  0])

In [122]:
# print shapes
print(lhs_exogenous_df.shape)
print(lhs_levers_df.shape)

(3000, 16)
(3000, 71)


In [123]:
lhs_df_merged = pd.merge(lhs_exogenous_df, lhs_levers_df, on=["region", "design_id", "future_id"], how="outer", suffixes=('_X', '_L'))
lhs_df_merged.head()

,region,design_id,future_id,46,47,48,49,50,51,52,...,1759,1760,1765,1766,1768,1776,1778,1790,1791,1799
0,louisiana,-1,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,...,0.644272,0.214180,0.955481,0.733736,0.116532,0.828401,0.217715,0.589410,0.939430,0.804396
1,louisiana,-1,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,...,0.486423,0.278023,0.680104,0.626837,0.392910,0.005622,0.342793,0.767436,0.435921,0.631640
2,louisiana,-1,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,...,0.331039,0.193177,0.968159,0.467609,0.402445,0.099340,0.676226,0.109182,0.582023,0.877888
3,louisiana,-1,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,...,0.267909,0.674077,0.246089,0.464519,0.968246,0.733699,0.143231,0.289890,0.794136,0.147144
4,louisiana,-1,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,...,0.135390,0.111915,0.455861,0.378087,0.267354,0.528790,0.529280,0.660236,0.690165,0.539415


In [124]:
# Filter the lhs_df_merged to only include rows where design_id is 4
lhs_df_merged = lhs_df_merged[lhs_df_merged.design_id == 4]
lhs_df_merged.shape

(1000, 84)

In [125]:
lhs_df_merged.design_id.unique()

array([4])

In [126]:
# NOTE: check col names, there should be no duplicates
lhs_df_merged.columns

Index(['region', 'design_id', 'future_id', '46', '47', '48', '49', '50', '51',
       '52', '53', '54', '55', '56', '57', '58', '1', '2', '3', '4', '5', '6',
       '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18',
       '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30',
       '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42',
       '43', '44', '45', '893', '1481', '1482', '1712', '1713', '1716', '1718',
       '1721', '1738', '1739', '1742', '1745', '1747', '1759', '1760', '1765',
       '1766', '1768', '1776', '1778', '1790', '1791', '1799'],
      dtype='object')

In [127]:
lhs_df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 2000 to 2999
Data columns (total 84 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   region     1000 non-null   object 
 1   design_id  1000 non-null   int64  
 2   future_id  1000 non-null   int64  
 3   46         1000 non-null   float64
 4   47         1000 non-null   float64
 5   48         1000 non-null   float64
 6   49         1000 non-null   float64
 7   50         1000 non-null   float64
 8   51         1000 non-null   float64
 9   52         1000 non-null   float64
 10  53         1000 non-null   float64
 11  54         1000 non-null   float64
 12  55         1000 non-null   float64
 13  56         1000 non-null   float64
 14  57         1000 non-null   float64
 15  58         1000 non-null   float64
 16  1          1000 non-null   float64
 17  2          1000 non-null   float64
 18  3          1000 non-null   float64
 19  4          1000 non-null   float64
 20  5         

## Load SISEPUEDE WIDE_INPUTS_OUTPUTS

In [128]:
attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY.csv"))
attr_primary_df

,primary_id,design_id,strategy_id,future_id
0,332332,4,0,0
1,402402,4,6004,0
2,402403,4,6004,1
3,402404,4,6004,2
4,402405,4,6004,3
...,...,...,...,...
2072,78078,0,6012,0
2073,79079,0,6013,0
2074,80080,0,6014,0
2075,81081,0,6015,0


In [129]:
attr_primary_df = attr_primary_df[attr_primary_df["strategy_id"].isin([6004])]
attr_primary_df

,primary_id,design_id,strategy_id,future_id
1,402402,4,6004,0
2,402403,4,6004,1
3,402404,4,6004,2
4,402405,4,6004,3
5,402406,4,6004,4
...,...,...,...,...
997,403398,4,6004,996
998,403399,4,6004,997
999,403400,4,6004,998
1000,403401,4,6004,999


In [130]:
wide_inputs_outputs_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "sisepuede_results_IDE_2025-09-18t09;19;22.726476.csv"))
wide_inputs_outputs_df

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne,design_id,strategy_id,future_id
0,1001,louisiana,7,0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,1.119173e+06,...,0,1204.745945,2.243801e+06,2.446506e+07,171684.426729,153.672261,0,0,1000,0
1,1001,louisiana,8,0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,1.114548e+06,...,0,928.252899,2.294739e+06,2.312859e+07,170974.875290,153.037151,0,0,1000,0
2,1001,louisiana,9,0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,1.109930e+06,...,0,1174.618939,2.271010e+06,2.470988e+07,170266.477753,152.403075,0,0,1000,0
3,1001,louisiana,10,0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,1.105320e+06,...,0,1169.740125,2.261577e+06,2.460725e+07,169559.271068,151.770064,0,0,1000,0
4,1001,louisiana,11,0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,1.100718e+06,...,0,1164.869776,2.252161e+06,2.450479e+07,168853.291338,151.138152,0,0,1000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55849,405404,louisiana,31,0,400576.538479,67862.606480,94.798899,92803.361279,8395.070116,1.324963e+06,...,0,1155.352512,2.540459e+06,2.354681e+07,197810.788975,175.631833,0,4,6006,1000
55850,405404,louisiana,32,0,403345.490574,68301.283110,95.578140,93476.074424,8466.096654,1.334500e+06,...,0,1162.560049,2.553451e+06,2.362574e+07,199386.653492,177.024429,0,4,6006,1000
55851,405404,louisiana,33,0,405982.273125,68728.197688,96.316704,94107.258753,8532.817098,1.343489e+06,...,0,1169.676600,2.565517e+06,2.370233e+07,200895.110392,178.359470,0,4,6006,1000
55852,405404,louisiana,34,0,408514.005639,69144.587083,97.024119,94706.642930,8596.297503,1.352054e+06,...,0,1176.688661,2.576854e+06,2.377645e+07,202350.521090,179.648983,0,4,6006,1000


In [131]:
wide_inputs_outputs_df['primary_id'].unique()

array([  1001,   2002,   3003, ..., 405402, 405403, 405404])

In [132]:
wide_inputs_outputs_df = wide_inputs_outputs_df[wide_inputs_outputs_df["primary_id"].isin(attr_primary_df["primary_id"].unique())]

In [133]:
wide_inputs_outputs_df['primary_id'].unique()

array([402402, 402403, 402404, 402405, 402406, 402407, 402408, 402409,
       402410, 402411, 402412, 402413, 402414, 402415, 402416, 402417,
       402418, 402419, 402420, 402422, 402423, 402424, 402425, 402426,
       402427, 402428, 402429, 402430, 402431, 402432, 402433, 402434,
       402435, 402436, 402437, 402438, 402440, 402441, 402442, 402443,
       402444, 402445, 402446, 402447, 402448, 402449, 402450, 402451,
       402452, 402453, 402454, 402455, 402456, 402457, 402458, 402459,
       402460, 402461, 402462, 402463, 402464, 402465, 402466, 402467,
       402468, 402469, 402470, 402471, 402472, 402473, 402474, 402475,
       402476, 402477, 402478, 402479, 402480, 402481, 402482, 402483,
       402484, 402485, 402486, 402487, 402488, 402489, 402491, 402492,
       402493, 402494, 402495, 402496, 402497, 402498, 402499, 402500,
       402501, 402502, 402503, 402504, 402505, 402506, 402507, 402508,
       402509, 402510, 402511, 402512, 402513, 402514, 402515, 402516,
      

In [134]:
wide_inputs_outputs_df.primary_id.nunique()

964

## Load Costs-Benefits Data

In [135]:
cb_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "wide_cb_data_lhc1000_2025_09_18.csv"))
cb_df


,primary_id,future_id,strategy_code,Year,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,...,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,402402,0,PFLO:ALL_LA_ACTIONS,2022,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000
1,402402,0,PFLO:ALL_LA_ACTIONS,2023,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000
2,402402,0,PFLO:ALL_LA_ACTIONS,2024,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000
3,402402,0,PFLO:ALL_LA_ACTIONS,2025,-3.440800e-16,-1.451481e-23,0.021749,0.000000e+00,6.959355e-12,0.000000,...,0.019772,0.000000,0.000000e+00,0.000000,-2.494771e-23,-5.151435e-14,0.000000e+00,-0.049444,0.000000e+00,0.000000
4,402402,0,PFLO:ALL_LA_ACTIONS,2026,-2.434978e-16,0.000000e+00,0.043782,2.673078e-14,7.691665e-12,0.000000,...,0.039802,0.000000,-5.968559e-17,0.000000,0.000000e+00,-1.098499e-14,-5.269051e-15,-0.100030,8.114148e-17,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53674,405404,1000,PFLO:ALL_LA_ACTIONS_LC,2046,1.563896e+00,3.082171e+00,0.702474,3.308030e+00,-4.519265e-02,-0.003608,...,0.421736,0.086861,-3.436004e-03,-0.508869,4.832892e+00,4.291144e+04,2.388970e+02,-7.557733,-1.470104e+00,-0.102172
53675,405404,1000,PFLO:ALL_LA_ACTIONS_LC,2047,1.631399e+00,3.291060e+00,0.737395,3.499125e+00,-4.462654e-02,-0.003815,...,0.441483,0.098313,-3.577373e-03,-0.536253,5.169280e+00,4.481848e+04,2.547007e+02,-8.112787,-1.628943e+00,-0.107289
53676,405404,1000,PFLO:ALL_LA_ACTIONS_LC,2048,1.702816e+00,3.508248e+00,0.772329,3.684007e+00,-4.376639e-02,-0.004023,...,0.461289,0.110385,-3.709618e-03,-0.564001,5.519847e+00,4.665187e+04,2.708138e+02,-8.711067,-1.798208e+00,-0.112407
53677,405404,1000,PFLO:ALL_LA_ACTIONS_LC,2049,1.780561e+00,3.734210e+00,0.807309,3.863807e+00,-4.264118e-02,-0.004233,...,0.481158,0.123094,-3.834383e-03,-0.592080,5.885389e+00,4.839787e+04,2.872174e+02,-9.433411,-1.978513e+00,-0.117526


In [ ]:
cb_df.future_id.nunique()

In [136]:
cb_df.primary_id.nunique()

1851

In [137]:
cb_df = cb_df[cb_df["primary_id"].isin(attr_primary_df["primary_id"].unique())]


In [138]:
cb_df.primary_id.nunique()

964

## Load LSU data

In [139]:
lsu_data = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "lsu_output_1000_ensemble.csv"))
lsu_data.head()

,time,primary_id,la_value_direct,la_value_indirect,la_value_induced,la_earnings_direct,la_earnings_indirect,la_earnings_induced,la_employment_direct,la_employment_indirect,la_employment_total
0,6,1001,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
1,7,1001,-3.306968e+07,-1.443522e+07,5.185341e+06,-2.004362e+07,-7.401543e+06,2.750115e+06,-305.470157,-115.361168,-360.375286
2,8,1001,-3.251623e+07,-1.415834e+07,3.974986e+06,-1.958543e+07,-7.264112e+06,2.107956e+06,-298.048791,-113.145168,-364.861314
3,9,1001,-3.553823e+07,-1.566130e+07,4.277098e+05,-2.208807e+07,-8.010408e+06,2.258517e+05,-338.544893,-125.180713,-458.793096
4,10,1001,-3.636556e+07,-1.607516e+07,-1.626845e+06,-2.278197e+07,-8.215295e+06,-8.642337e+05,-349.814745,-128.492196,-497.351563


In [140]:
lsu_data = lsu_data[lsu_data["primary_id"].isin(attr_primary_df["primary_id"].unique())]

In [141]:
lsu_data.primary_id.nunique()

964

In [142]:
lsu_data=lsu_data.rename(columns={"la_employment_total": "num_jobs"})

In [143]:
lsu_data

,time,primary_id,la_value_direct,la_value_indirect,la_value_induced,la_earnings_direct,la_earnings_indirect,la_earnings_induced,la_employment_direct,la_employment_indirect,num_jobs
2220,6,402402,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
2221,7,402402,-3.306968e+07,-1.443522e+07,5.185341e+06,-2.004362e+07,-7.401543e+06,2.750115e+06,-305.470157,-115.361168,-360.375286
2222,8,402402,-3.251623e+07,-1.415834e+07,3.974986e+06,-1.958543e+07,-7.264112e+06,2.107956e+06,-298.048791,-113.145168,-364.861314
2223,9,402402,-3.553823e+07,-1.566130e+07,4.277098e+05,-2.208807e+07,-8.010408e+06,2.258517e+05,-338.544893,-125.180713,-458.793096
2224,10,402402,-3.636556e+07,-1.607516e+07,-1.626845e+06,-2.278197e+07,-8.215295e+06,-8.642337e+05,-349.814745,-128.492196,-497.351563
...,...,...,...,...,...,...,...,...,...,...,...
31135,31,403402,1.296619e+09,4.064799e+08,-9.939956e+08,8.333718e+08,1.988632e+08,-5.274363e+08,10912.087438,2999.584598,2317.867683
31136,32,403402,1.415108e+09,4.406630e+08,-1.086097e+09,8.859286e+08,2.181457e+08,-5.763044e+08,11551.255098,3280.785929,2163.816326
31137,33,403402,1.582065e+09,4.882299e+08,-1.175086e+09,9.637764e+08,2.450804e+08,-6.235225e+08,12524.829900,3675.816040,2494.590523
31138,34,403402,1.690440e+09,5.193217e+08,-1.302572e+09,1.008360e+09,2.628271e+08,-6.911636e+08,13049.567373,3932.737217,1788.756742


In [145]:
lsu_data["num_jobs"].describe()

count    28920.000000
mean     15480.581271
std      17002.516262
min     -60922.548872
25%          0.000000
50%      13524.589073
75%      27693.623529
max      97942.985211
Name: num_jobs, dtype: float64

In [144]:
lsu_data["earning_per_job"] = (
    lsu_data[["la_earnings_direct", "la_earnings_indirect", "la_earnings_induced"]].sum(axis=1, min_count=1)
    / lsu_data["num_jobs"].replace(0, pd.NA)
)


In [146]:
lsu_data["earning_per_job"].describe()

count     27956.000000
unique    25108.000000
top       68520.213048
freq        660.000000
Name: earning_per_job, dtype: float64

In [147]:
lsu_data.primary_id.nunique()

964

In [148]:
lsu_data

,time,primary_id,la_value_direct,la_value_indirect,la_value_induced,la_earnings_direct,la_earnings_indirect,la_earnings_induced,la_employment_direct,la_employment_indirect,num_jobs,earning_per_job
2220,6,402402,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,<NA>
2221,7,402402,-3.306968e+07,-1.443522e+07,5.185341e+06,-2.004362e+07,-7.401543e+06,2.750115e+06,-305.470157,-115.361168,-360.375286,68525.910196
2222,8,402402,-3.251623e+07,-1.415834e+07,3.974986e+06,-1.958543e+07,-7.264112e+06,2.107956e+06,-298.048791,-113.145168,-364.861314,67810.925884
2223,9,402402,-3.553823e+07,-1.566130e+07,4.277098e+05,-2.208807e+07,-8.010408e+06,2.258517e+05,-338.544893,-125.180713,-458.793096,65111.322511
2224,10,402402,-3.636556e+07,-1.607516e+07,-1.626845e+06,-2.278197e+07,-8.215295e+06,-8.642337e+05,-349.814745,-128.492196,-497.351563,64062.325075
...,...,...,...,...,...,...,...,...,...,...,...,...
31135,31,403402,1.296619e+09,4.064799e+08,-9.939956e+08,8.333718e+08,1.988632e+08,-5.274363e+08,10912.087438,2999.584598,2317.867683,217785.844868
31136,32,403402,1.415108e+09,4.406630e+08,-1.086097e+09,8.859286e+08,2.181457e+08,-5.763044e+08,11551.255098,3280.785929,2163.816326,243906.959672
31137,33,403402,1.582065e+09,4.882299e+08,-1.175086e+09,9.637764e+08,2.450804e+08,-6.235225e+08,12524.829900,3675.816040,2494.590523,234641.418951
31138,34,403402,1.690440e+09,5.193217e+08,-1.302572e+09,1.008360e+09,2.628271e+08,-6.911636e+08,13049.567373,3932.737217,1788.756742,324260.472991


## Data Cleaning

### SISEPUEDE Emission data

In [149]:
# Get the subsector total variables
subsector_total_vars = [c for c in wide_inputs_outputs_df.columns if "emission_co2e_subsector_total" in c]

In [150]:
# Filter to only subsector total columns and primary_id, time_period
la_emissions_df = wide_inputs_outputs_df[["primary_id", "time_period"] + subsector_total_vars]
la_emissions_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
2146,402402,7,2.924249,0.0,35.431379,13.094104,-36.545088,117.042622,2.762559,-0.077001,0.230319,1.539811,4.491483,1.233181,45.130223,0.447204,3.138402
2147,402402,8,2.864242,0.0,34.534840,13.210824,-39.679345,114.912120,2.773548,-0.104854,0.229389,1.516623,4.525427,1.226921,45.865290,0.454013,3.189568
2148,402402,9,2.912181,0.0,33.894560,13.446147,-42.096028,115.129312,2.786645,-0.132605,0.228545,1.493794,4.561015,1.210181,46.698636,0.461144,3.239706
2149,402402,10,2.900085,0.0,33.257178,13.409080,-44.041357,114.908846,2.801782,-0.160253,0.227765,1.471320,4.598400,1.181498,47.611029,0.468528,3.291336
2150,402402,11,2.888010,0.0,33.417577,13.446944,-45.670588,114.346550,2.818877,-0.187795,0.227044,1.449198,4.637649,1.140642,48.587463,0.476114,3.343789


In [151]:
la_emissions_df.tail()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
30097,403402,31,3.112036,-2.238299,30.738089,11.067723,-61.421998,102.105669,2.337172,-2.633551,0.240192,1.582927,4.258078,2.489190,36.047402,0.604055,4.761125
30098,403402,32,3.121138,-2.356105,31.394516,10.422884,-62.378133,101.116021,2.304533,-2.789660,0.239433,1.577197,4.263980,2.421315,35.601239,0.611832,4.842982
30099,403402,33,3.129091,-2.473910,32.131378,9.777965,-63.357448,100.165656,2.271600,-2.920143,0.238481,1.569676,4.272715,2.361479,35.146380,0.619750,4.925968
30100,403402,34,3.136147,-2.591715,32.905735,9.132837,-64.356744,99.254228,2.238244,-3.024123,0.237362,1.560663,4.284121,2.257373,34.676987,0.627807,5.010081
30101,403402,35,3.142522,-2.709520,33.713830,8.492052,-65.373837,98.486559,2.204311,-3.039465,0.236096,1.550412,4.298122,2.154360,34.186146,0.635996,5.095282


### Production Data

In [152]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']

In [153]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [154]:
# 5) Loop over fuels and sectors
# Use the original dataframe directly
# Initialize results dataframe
ind_fuel_demand_by_sector = pd.DataFrame({
    'primary_id': wide_inputs_outputs_df['primary_id'],
    'time_period': wide_inputs_outputs_df['time_period']
}, index=wide_inputs_outputs_df.index)

# Loop over fuels and sectors
for fuel in relevant_fuels:
    # efficiency column for this fuel
    eff_cols = [c for c in wide_inputs_outputs_df.columns
                if c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{fuel}')]
    if not eff_cols:
        continue
    fuel_efficiency = wide_inputs_outputs_df[eff_cols[0]]

    for sector in sectors:
        sector_dem_cols = [c for c in wide_inputs_outputs_df.columns
                           if f'energy_demand_inen_{sector}' in c]
        sector_fuel_fraction_cols = [c for c in wide_inputs_outputs_df.columns
                                     if f'frac_inen_energy_{sector}_{fuel}' in c]

        if sector_dem_cols and sector_fuel_fraction_cols:
            sector_total_demand = wide_inputs_outputs_df[sector_dem_cols[0]]
            sector_fuel_fraction = wide_inputs_outputs_df[sector_fuel_fraction_cols[0]]

            if (sector_fuel_fraction * sector_total_demand).sum() > 0 or fuel == 'electricity':
                sector_fuel_demand = sector_fuel_fraction * sector_total_demand
                ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand

                # CAPEX/OPEX
                if fuel == 'electricity':
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_electricity
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_electricity
                    )
                else:
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_other
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_other
                    )

                # Fuel consumed
                sector_fuel_consumed = sector_fuel_demand / fuel_efficiency

                # Baseline = first time_period per primary_id
                sector_fuel_consumed_baseline = (
                    sector_fuel_consumed.groupby(wide_inputs_outputs_df['primary_id'])
                                        .transform('first')
                )

                # Change relative to baseline
                sector_change_in_fuel_consumed = (
                    sector_fuel_consumed_baseline - sector_fuel_consumed
                )

                # Save results
                ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed
                )
                ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * capex_multiplier_efficiency
                )
                ind_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * opex_multiplier_efficiency
                )


C:\Users\nasta\AppData\Local\Temp\ipykernel_168852\3509172589.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = (
C:\Users\nasta\AppData\Local\Temp\ipykernel_168852\3509172589.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
C:\Users\nasta\AppData\Local\Temp\ipykernel_168852\3509172589.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f

In [155]:
ind_fuel_demand_by_sector

,primary_id,time_period,energy_demand_cement_coal,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,efficiency_energy_saving_cement_coal,efficiency_capex_cement_coal,efficiency_opex_cement_coal,energy_demand_chemicals_coal,energy_demand_capex_chemicals_coal,...,energy_demand_opex_other_product_manufacturing_oil,efficiency_energy_saving_other_product_manufacturing_oil,efficiency_capex_other_product_manufacturing_oil,efficiency_opex_other_product_manufacturing_oil,energy_demand_recycled_wood_oil,energy_demand_capex_recycled_wood_oil,energy_demand_opex_recycled_wood_oil,efficiency_energy_saving_recycled_wood_oil,efficiency_capex_recycled_wood_oil,efficiency_opex_recycled_wood_oil
2146,402402,7,15.110409,1.680276e+07,6.301036e+06,0.000000,0.000000e+00,0.0,0.000732,813.906668,...,1.065403e+07,0.000000,0.000000e+00,0.0,0.319755,355567.121778,133337.670667,0.000000,0.000000e+00,0.0
2147,402402,8,12.290454,1.366697e+07,5.125115e+06,4.762860,4.762860e+07,0.0,0.000070,77.506444,...,1.101825e+07,-1.019247,-1.019247e+07,-0.0,0.314100,349279.179225,130979.692209,0.009181,9.181124e+04,0.0
2148,402402,9,11.835264,1.316080e+07,4.935302e+06,5.593867,5.593867e+07,0.0,0.000000,0.000000,...,1.138607e+07,-2.041192,-2.041192e+07,-0.0,0.306088,340369.779635,127638.667363,0.021401,2.140106e+05,0.0
2149,402402,10,11.597945,1.289691e+07,4.836340e+06,6.061209,6.061209e+07,0.0,0.000000,0.000000,...,1.179935e+07,-3.197407,-3.197407e+07,-0.0,0.297894,331257.757247,124221.658968,0.033769,3.376881e+05,0.0
2150,402402,11,10.757587,1.196243e+07,4.485911e+06,7.510201,7.510201e+07,0.0,0.000000,0.000000,...,1.222929e+07,-4.395974,-4.395974e+07,-0.0,0.290604,323151.473668,121181.802625,0.044862,4.486193e+05,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30097,403402,31,0.000000,0.000000e+00,0.000000e+00,25.083076,2.508308e+08,0.0,0.000000,0.000000,...,9.760938e+06,10.278636,1.027864e+08,0.0,0.051724,57516.981125,21568.867922,0.372370,3.723702e+06,0.0
30098,403402,32,0.000000,0.000000e+00,0.000000e+00,25.083076,2.508308e+08,0.0,0.000000,0.000000,...,9.486657e+06,11.172821,1.117282e+08,0.0,0.046466,51669.845870,19376.192201,0.378152,3.781519e+06,0.0
30099,403402,33,0.000000,0.000000e+00,0.000000e+00,25.083076,2.508308e+08,0.0,0.000000,0.000000,...,9.204921e+06,12.063254,1.206325e+08,0.0,0.041599,46258.202991,17346.826122,0.383424,3.834244e+06,0.0
30100,403402,34,0.000000,0.000000e+00,0.000000e+00,25.083076,2.508308e+08,0.0,0.000000,0.000000,...,8.915359e+06,12.951351,1.295135e+08,0.0,0.037098,41253.268976,15469.975866,0.388230,3.882302e+06,0.0


In [156]:
ind_fuel_demand_by_sector.primary_id.nunique()

964

In [157]:
ind_fuel_demand_by_sector.isna().sum().sum() 

np.int64(0)

### CB Data

In [158]:
# Make all column names lowercase
cb_df.columns = [c.lower() for c in cb_df.columns]

# Filter to only important cb columns
cb_df = cb_df[["primary_id",
               "future_id",
               "year",
               "technical_cost",
               #"consumer_savings",
               #"human_health",
               "air_pollution"]]

cb_df.head()

,primary_id,future_id,year,technical_cost,air_pollution
0,402402,0,2022,0.000000,0.000000e+00
1,402402,0,2023,0.000000,0.000000e+00
2,402402,0,2024,0.000000,0.000000e+00
3,402402,0,2025,-0.049444,-3.440800e-16
4,402402,0,2026,-0.100030,-2.434978e-16


In [159]:
cb_df.tail()

,primary_id,future_id,year,technical_cost,air_pollution
27951,403402,1000,2046,-8.892776,1.532292
27952,403402,1000,2047,-9.120479,1.600508
27953,403402,1000,2048,-9.985812,1.675685
27954,403402,1000,2049,-10.671408,1.742389
27955,403402,1000,2050,-11.417159,1.821379


In [160]:
cb_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27956 entries, 0 to 27955
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   primary_id      27956 non-null  int64  
 1   future_id       27956 non-null  int64  
 2   year            27956 non-null  int64  
 3   technical_cost  27956 non-null  float64
 4   air_pollution   27956 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 1.3 MB


### LSU data

In [161]:
lsu_data = lsu_data[["primary_id",
               "time",
               "num_jobs",
               "earning_per_job"]]

lsu_data.head()

,primary_id,time,num_jobs,earning_per_job
2220,402402,6,0.000000,<NA>
2221,402402,7,-360.375286,68525.910196
2222,402402,8,-364.861314,67810.925884
2223,402402,9,-458.793096,65111.322511
2224,402402,10,-497.351563,64062.325075


In [162]:
lsu_data= lsu_data.dropna()

In [163]:
lsu_data

,primary_id,time,num_jobs,earning_per_job
2221,402402,7,-360.375286,68525.910196
2222,402402,8,-364.861314,67810.925884
2223,402402,9,-458.793096,65111.322511
2224,402402,10,-497.351563,64062.325075
2225,402402,11,-482.114419,63014.838826
...,...,...,...,...
31135,403402,31,2317.867683,217785.844868
31136,403402,32,2163.816326,243906.959672
31137,403402,33,2494.590523,234641.418951
31138,403402,34,1788.756742,324260.472991


In [164]:
lsu_data['earning_per_job'].describe()

count     27956.000000
unique    25108.000000
top       68520.213048
freq        660.000000
Name: earning_per_job, dtype: float64

## LHS Data

In [165]:
lhs_df_merged = lhs_df_merged.drop(columns=["design_id", "region"])
lhs_df_merged.head()

,future_id,46,47,48,49,50,51,52,53,54,...,1759,1760,1765,1766,1768,1776,1778,1790,1791,1799
2000,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,0.508737,...,0.644272,0.214180,0.955481,0.733736,0.116532,0.828401,0.217715,0.589410,0.939430,0.804396
2001,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,0.973614,...,0.486423,0.278023,0.680104,0.626837,0.392910,0.005622,0.342793,0.767436,0.435921,0.631640
2002,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,0.053175,...,0.331039,0.193177,0.968159,0.467609,0.402445,0.099340,0.676226,0.109182,0.582023,0.877888
2003,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,0.885947,...,0.267909,0.674077,0.246089,0.464519,0.968246,0.733699,0.143231,0.289890,0.794136,0.147144
2004,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,0.223495,...,0.135390,0.111915,0.455861,0.378087,0.267354,0.528790,0.529280,0.660236,0.690165,0.539415


## Transform time series format into single-row format

### SISEPUEDE Emission data

In [166]:
# Sum all the subsector emission columns across axis=1
la_emission_total_df = la_emissions_df.copy()
la_emission_total_df["emission_total"] = la_emission_total_df[subsector_total_vars].sum(axis=1)
la_emission_total_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso,emission_total
2146,402402,7,2.924249,0.0,35.431379,13.094104,-36.545088,117.042622,2.762559,-0.077001,0.230319,1.539811,4.491483,1.233181,45.130223,0.447204,3.138402,190.843446
2147,402402,8,2.864242,0.0,34.534840,13.210824,-39.679345,114.912120,2.773548,-0.104854,0.229389,1.516623,4.525427,1.226921,45.865290,0.454013,3.189568,185.518606
2148,402402,9,2.912181,0.0,33.894560,13.446147,-42.096028,115.129312,2.786645,-0.132605,0.228545,1.493794,4.561015,1.210181,46.698636,0.461144,3.239706,183.833231
2149,402402,10,2.900085,0.0,33.257178,13.409080,-44.041357,114.908846,2.801782,-0.160253,0.227765,1.471320,4.598400,1.181498,47.611029,0.468528,3.291336,181.925237
2150,402402,11,2.888010,0.0,33.417577,13.446944,-45.670588,114.346550,2.818877,-0.187795,0.227044,1.449198,4.637649,1.140642,48.587463,0.476114,3.343789,180.921474


In [167]:
# Keep only the primary_id, time_period, and emission_total columns
la_emission_total_df = la_emission_total_df[["primary_id", "time_period", "emission_total"]]
la_emission_total_df.head()

,primary_id,time_period,emission_total
2146,402402,7,190.843446
2147,402402,8,185.518606
2148,402402,9,183.833231
2149,402402,10,181.925237
2150,402402,11,180.921474


In [168]:
la_emission_total_df.tail()

,primary_id,time_period,emission_total
30097,403402,31,133.049810
30098,403402,32,130.393172
30099,403402,33,127.858639
30100,403402,34,125.349002
30101,403402,35,123.072864


### Emission data sum

In [169]:
# aggregate data by primary_id summing the emissions
la_emission_df_sum_agg = la_emission_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_emission_df_sum_agg = la_emission_df_sum_agg.drop(columns=["time_period"])
la_emission_df_sum_agg.head()

,primary_id,emission_total
0,402402,3308.232823
1,402403,4713.726381
2,402404,5168.255434
3,402405,4411.805579
4,402406,5094.995066


### Emission data mean

In [170]:
# Filter out rows with time_period < 31
la_filtered_emission_total_df = la_emission_total_df[la_emission_total_df["time_period"] >= 31]
la_filtered_emission_total_df = la_filtered_emission_total_df.reset_index(drop=True)
la_filtered_emission_total_df.head(7)

,primary_id,time_period,emission_total
0,402402,31,36.224970
1,402402,32,31.475651
2,402402,33,26.309689
3,402402,34,20.816626
4,402402,35,15.660846
5,402403,31,138.936315
6,402403,32,136.741637


In [171]:
# aggregate data by primary_id by summing the emissions
la_emission_df_mean_agg = la_filtered_emission_total_df.groupby(["primary_id"]).mean().reset_index()

# Rename emission_total to emission_avg_last_five_years
la_emission_df_mean_agg.rename(columns={"emission_total": "emission_avg_last_five_years"}, inplace=True)
la_emission_df_mean_agg

,primary_id,time_period,emission_avg_last_five_years
0,402402,33.0,26.097556
1,402403,33.0,134.250713
2,402404,33.0,164.387644
3,402405,33.0,104.151002
4,402406,33.0,157.680453
...,...,...,...
959,403398,33.0,121.594420
960,403399,33.0,162.146717
961,403400,33.0,194.313061
962,403401,33.0,200.981586


In [172]:
# Drop year column as it is no longer needed
la_emission_df_mean_agg = la_emission_df_mean_agg.drop(columns=["time_period"])
la_emission_df_mean_agg.head()

,primary_id,emission_avg_last_five_years
0,402402,26.097556
1,402403,134.250713
2,402404,164.387644
3,402405,104.151002
4,402406,157.680453


### Combining emission agg into one df

In [173]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)
print("la_emission_df_sum_agg shape:", la_emission_df_sum_agg.shape)

la_emission_df_mean_agg shape: (964, 2)
la_emission_df_sum_agg shape: (964, 2)


In [174]:
la_emissions_df_merged = la_emission_df_mean_agg.merge(la_emission_df_sum_agg, on="primary_id", how="inner")
la_emissions_df_merged.head()

,primary_id,emission_avg_last_five_years,emission_total
0,402402,26.097556,3308.232823
1,402403,134.250713,4713.726381
2,402404,164.387644,5168.255434
3,402405,104.151002,4411.805579
4,402406,157.680453,5094.995066


In [175]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)

la_emission_df_mean_agg shape: (964, 2)


### Production and Production Cost Data

In [176]:
industry_cost_vars = [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_capex_") or c.startswith("energy_demand_opex_")
]

In [177]:
industry_product_vars =  [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_")
    and "_capex_" not in c
    and "_opex_" not in c
]

In [178]:
ind_fuel_demand_by_sector[industry_cost_vars]

,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,energy_demand_capex_chemicals_coal,energy_demand_opex_chemicals_coal,energy_demand_capex_lime_and_carbonite_coal,energy_demand_opex_lime_and_carbonite_coal,energy_demand_capex_mining_coal,energy_demand_opex_mining_coal,energy_demand_capex_other_product_manufacturing_coal,energy_demand_opex_other_product_manufacturing_coal,...,energy_demand_capex_electronics_oil,energy_demand_opex_electronics_oil,energy_demand_capex_lime_and_carbonite_oil,energy_demand_opex_lime_and_carbonite_oil,energy_demand_capex_mining_oil,energy_demand_opex_mining_oil,energy_demand_capex_other_product_manufacturing_oil,energy_demand_opex_other_product_manufacturing_oil,energy_demand_capex_recycled_wood_oil,energy_demand_opex_recycled_wood_oil
2146,1.680276e+07,6.301036e+06,813.906668,305.215000,0.0,0.0,174.208320,65.328120,3.593569e+06,1.347588e+06,...,1694.069129,635.275924,0.000000,0.000000,12954.468460,4857.925672,2.841075e+07,1.065403e+07,355567.121778,133337.670667
2147,1.366697e+07,5.125115e+06,77.506444,29.064916,0.0,0.0,139.114963,52.168111,3.312895e+06,1.242336e+06,...,1790.850816,671.569056,0.000000,0.000000,12061.561422,4523.085533,2.938199e+07,1.101825e+07,349279.179225,130979.692209
2148,1.316080e+07,4.935302e+06,0.000000,0.000000,0.0,0.0,107.199600,40.199850,3.055281e+06,1.145730e+06,...,1830.345443,686.379541,0.000000,0.000000,11109.988295,4166.245611,3.036286e+07,1.138607e+07,340369.779635,127638.667363
2149,1.289691e+07,4.836340e+06,0.000000,0.000000,0.0,0.0,70.711454,26.516795,2.790288e+06,1.046358e+06,...,1900.179145,712.567179,0.000000,0.000000,10088.438813,3783.164555,3.146494e+07,1.179935e+07,331257.757247,124221.658968
2150,1.196243e+07,4.485911e+06,0.000000,0.000000,0.0,0.0,26.523142,9.946178,2.540861e+06,9.528230e+05,...,1988.692454,745.759670,0.000000,0.000000,9001.151673,3375.431877,3.261144e+07,1.222929e+07,323151.473668,121181.802625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30097,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000e+00,0.000000e+00,...,2579.986185,967.494819,274786.972929,103045.114848,0.000000,0.000000,2.602917e+07,9.760938e+06,57516.981125,21568.867922
30098,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000e+00,0.000000e+00,...,2525.211972,946.954490,260530.897523,97699.086571,0.000000,0.000000,2.529775e+07,9.486657e+06,51669.845870,19376.192201
30099,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000e+00,0.000000e+00,...,2468.804057,925.801521,246406.627950,92402.485481,0.000000,0.000000,2.454646e+07,9.204921e+06,46258.202991,17346.826122
30100,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000e+00,0.000000e+00,...,2410.458918,903.922094,232438.588102,87164.470538,0.000000,0.000000,2.377429e+07,8.915359e+06,41253.268976,15469.975866


In [179]:
ind_fuel_demand_by_sector[industry_product_vars]

,energy_demand_cement_coal,energy_demand_chemicals_coal,energy_demand_lime_and_carbonite_coal,energy_demand_mining_coal,energy_demand_other_product_manufacturing_coal,energy_demand_recycled_wood_coal,energy_demand_cement_coke,energy_demand_chemicals_coke,energy_demand_lime_and_carbonite_coke,energy_demand_mining_coke,...,energy_demand_recycled_wood_natural_gas,energy_demand_rubber_and_leather_natural_gas,energy_demand_textiles_natural_gas,energy_demand_cement_oil,energy_demand_chemicals_oil,energy_demand_electronics_oil,energy_demand_lime_and_carbonite_oil,energy_demand_mining_oil,energy_demand_other_product_manufacturing_oil,energy_demand_recycled_wood_oil
2146,15.110409,0.000732,0.0,0.000157,3.231629,0.006179,1.678934,0.000081,0.0,0.000017,...,1.542931,0.0,0.061508,12.726039,0.010728,0.001523,0.000000,0.011650,25.549257,0.319755
2147,12.290454,0.000070,0.0,0.000125,2.979224,0.006488,1.365606,0.000008,0.0,0.000014,...,1.507224,0.0,0.063520,13.489543,0.010654,0.001610,0.000000,0.010847,26.422671,0.314100
2148,11.835264,0.000000,0.0,0.000096,2.747557,0.005913,1.315029,0.000000,0.0,0.000011,...,1.502519,0.0,0.066175,12.966827,0.010629,0.001646,0.000000,0.009991,27.304749,0.306088
2149,11.597945,0.000000,0.0,0.000064,2.509253,0.004858,1.288661,0.000000,0.0,0.000007,...,1.524182,0.0,0.068386,12.342857,0.010597,0.001709,0.000000,0.009072,28.295832,0.297894
2150,10.757587,0.000000,0.0,0.000024,2.284949,0.003702,1.195287,0.000000,0.0,0.000003,...,1.562164,0.0,0.071145,12.182521,0.010575,0.001788,0.000000,0.008095,29.326858,0.290604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30097,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.752470,0.0,0.106250,9.866454,0.006147,0.002320,0.247111,0.000000,23.407543,0.051724
30098,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.717744,0.0,0.105365,9.657611,0.005965,0.002271,0.234291,0.000000,22.749793,0.046466
30099,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.683555,0.0,0.104397,9.447178,0.005786,0.002220,0.221589,0.000000,22.074167,0.041599
30100,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.649832,0.0,0.103326,9.234436,0.005609,0.002168,0.209028,0.000000,21.379774,0.037098


#### production cost

In [180]:
# Sum all the subsector emission columns across axis=1
la_production_cost_total_df = ind_fuel_demand_by_sector.copy()
la_production_cost_total_df["production_cost_total"] = la_production_cost_total_df[industry_cost_vars].sum(axis=1)

# Show only the inputs that went into the sum + the result
la_production_cost_total_df[industry_cost_vars + ["production_cost_total"]].head()


,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,energy_demand_capex_chemicals_coal,energy_demand_opex_chemicals_coal,energy_demand_capex_lime_and_carbonite_coal,energy_demand_opex_lime_and_carbonite_coal,energy_demand_capex_mining_coal,energy_demand_opex_mining_coal,energy_demand_capex_other_product_manufacturing_coal,energy_demand_opex_other_product_manufacturing_coal,...,energy_demand_opex_electronics_oil,energy_demand_capex_lime_and_carbonite_oil,energy_demand_opex_lime_and_carbonite_oil,energy_demand_capex_mining_oil,energy_demand_opex_mining_oil,energy_demand_capex_other_product_manufacturing_oil,energy_demand_opex_other_product_manufacturing_oil,energy_demand_capex_recycled_wood_oil,energy_demand_opex_recycled_wood_oil,production_cost_total
2146,1.680276e+07,6.301036e+06,813.906668,305.215000,0.0,0.0,174.208320,65.328120,3.593569e+06,1.347588e+06,...,635.275924,0.0,0.0,12954.468460,4857.925672,2.841075e+07,1.065403e+07,355567.121778,133337.670667,4.621730e+08
2147,1.366697e+07,5.125115e+06,77.506444,29.064916,0.0,0.0,139.114963,52.168111,3.312895e+06,1.242336e+06,...,671.569056,0.0,0.0,12061.561422,4523.085533,2.938199e+07,1.101825e+07,349279.179225,130979.692209,4.572407e+08
2148,1.316080e+07,4.935302e+06,0.000000,0.000000,0.0,0.0,107.199600,40.199850,3.055281e+06,1.145730e+06,...,686.379541,0.0,0.0,11109.988295,4166.245611,3.036286e+07,1.138607e+07,340369.779635,127638.667363,4.529696e+08
2149,1.289691e+07,4.836340e+06,0.000000,0.000000,0.0,0.0,70.711454,26.516795,2.790288e+06,1.046358e+06,...,712.567179,0.0,0.0,10088.438813,3783.164555,3.146494e+07,1.179935e+07,331257.757247,124221.658968,4.496721e+08
2150,1.196243e+07,4.485911e+06,0.000000,0.000000,0.0,0.0,26.523142,9.946178,2.540861e+06,9.528230e+05,...,745.759670,0.0,0.0,9001.151673,3375.431877,3.261144e+07,1.222929e+07,323151.473668,121181.802625,4.468016e+08


In [181]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_cost_total_df = la_production_cost_total_df[["primary_id", "time_period", "production_cost_total"]]
la_production_cost_total_df.head()

,primary_id,time_period,production_cost_total
2146,402402,7,4.621730e+08
2147,402402,8,4.572407e+08
2148,402402,9,4.529696e+08
2149,402402,10,4.496721e+08
2150,402402,11,4.468016e+08


In [182]:
# aggregate data by primary_id summing the emissions
la_production_cost_df_mean_agg = la_production_cost_total_df.groupby(["primary_id"]).mean().reset_index()

# Drop time period column
la_production_cost_df_mean_agg = la_production_cost_df_mean_agg.drop(columns=["time_period"])
la_production_cost_df_mean_agg.head()

,primary_id,production_cost_total
0,402402,4.540606e+08
1,402403,4.567414e+08
2,402404,4.708078e+08
3,402405,4.615890e+08
4,402406,4.799190e+08


In [183]:
la_production_cost_df_mean_agg.shape

(964, 2)

#### production

In [184]:
# columns: base energy demand only (no capex/opex)
industry_product_vars = [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_") and "_capex_" not in c and "_opex_" not in c
]

la_production_total_df = ind_fuel_demand_by_sector.copy()
la_production_total_df["production_total"] = (
    la_production_total_df[industry_product_vars].sum(axis=1, min_count=1)
)

# show only what went into the sum + the result
la_production_total_df[industry_product_vars + ["production_total"]].head()


,energy_demand_cement_coal,energy_demand_chemicals_coal,energy_demand_lime_and_carbonite_coal,energy_demand_mining_coal,energy_demand_other_product_manufacturing_coal,energy_demand_recycled_wood_coal,energy_demand_cement_coke,energy_demand_chemicals_coke,energy_demand_lime_and_carbonite_coke,energy_demand_mining_coke,...,energy_demand_rubber_and_leather_natural_gas,energy_demand_textiles_natural_gas,energy_demand_cement_oil,energy_demand_chemicals_oil,energy_demand_electronics_oil,energy_demand_lime_and_carbonite_oil,energy_demand_mining_oil,energy_demand_other_product_manufacturing_oil,energy_demand_recycled_wood_oil,production_total
2146,15.110409,0.000732,0.0,0.000157,3.231629,0.006179,1.678934,0.000081,0.0,0.000017,...,0.0,0.061508,12.726039,0.010728,0.001523,0.0,0.011650,25.549257,0.319755,275.853546
2147,12.290454,0.000070,0.0,0.000125,2.979224,0.006488,1.365606,0.000008,0.0,0.000014,...,0.0,0.063520,13.489543,0.010654,0.001610,0.0,0.010847,26.422671,0.314100,273.821482
2148,11.835264,0.000000,0.0,0.000096,2.747557,0.005913,1.315029,0.000000,0.0,0.000011,...,0.0,0.066175,12.966827,0.010629,0.001646,0.0,0.009991,27.304749,0.306088,272.055216
2149,11.597945,0.000000,0.0,0.000064,2.509253,0.004858,1.288661,0.000000,0.0,0.000007,...,0.0,0.068386,12.342857,0.010597,0.001709,0.0,0.009072,28.295832,0.297894,270.574072
2150,10.757587,0.000000,0.0,0.000024,2.284949,0.003702,1.195287,0.000000,0.0,0.000003,...,0.0,0.071145,12.182521,0.010575,0.001788,0.0,0.008095,29.326858,0.290604,269.223099


In [185]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_total_df = la_production_total_df[["primary_id", "time_period", "production_total"]]
la_production_total_df.head()

,primary_id,time_period,production_total
2146,402402,7,275.853546
2147,402402,8,273.821482
2148,402402,9,272.055216
2149,402402,10,270.574072
2150,402402,11,269.223099


In [186]:
# aggregate data by primary_id summing the emissions
la_production_df_mean_agg = la_production_total_df.groupby(["primary_id"]).mean().reset_index()

# Drop time period column
la_production_df_mean_agg = la_production_df_mean_agg.drop(columns=["time_period"])
la_production_df_mean_agg.head()

,primary_id,production_total
0,402402,251.650034
1,402403,279.612528
2,402404,276.395578
3,402405,264.412432
4,402406,275.458617


### CB data

In [187]:
# aggregate data by primary_id and region by summing the technical cost
cb_df_agg = cb_df.groupby(["primary_id", "future_id"]).mean().reset_index()
cb_df_agg

,primary_id,future_id,year,technical_cost,air_pollution
0,402402,0,2036.0,-8.906431,1.450083
1,402403,1,2036.0,-2.275659,1.033437
2,402404,2,2036.0,-7.468213,0.753391
3,402405,3,2036.0,-3.833560,1.181780
4,402406,4,2036.0,-10.618699,0.727183
...,...,...,...,...,...
959,403398,996,2036.0,-4.849271,0.827842
960,403399,997,2036.0,-8.026771,0.250140
961,403400,998,2036.0,-5.954081,0.030116
962,403401,999,2036.0,-7.731554,0.563744


In [188]:
# Drop year column as it is no longer needed
cb_df_agg = cb_df_agg.drop(columns=["year"], errors='ignore')
cb_df_agg.head()

,primary_id,future_id,technical_cost,air_pollution
0,402402,0,-8.906431,1.450083
1,402403,1,-2.275659,1.033437
2,402404,2,-7.468213,0.753391
3,402405,3,-3.833560,1.181780
4,402406,4,-10.618699,0.727183


In [189]:
cb_df_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 964 entries, 0 to 963
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   primary_id      964 non-null    int64  
 1   future_id       964 non-null    int64  
 2   technical_cost  964 non-null    float64
 3   air_pollution   964 non-null    float64
dtypes: float64(2), int64(2)
memory usage: 30.3 KB


### LSU data

In [190]:
lsu_data

,primary_id,time,num_jobs,earning_per_job
2221,402402,7,-360.375286,68525.910196
2222,402402,8,-364.861314,67810.925884
2223,402402,9,-458.793096,65111.322511
2224,402402,10,-497.351563,64062.325075
2225,402402,11,-482.114419,63014.838826
...,...,...,...,...
31135,403402,31,2317.867683,217785.844868
31136,403402,32,2163.816326,243906.959672
31137,403402,33,2494.590523,234641.418951
31138,403402,34,1788.756742,324260.472991


In [191]:
# aggregate data by primary_id and region by summing the technical cost
lsu_data_agg = lsu_data.groupby(["primary_id"]).mean().reset_index()
lsu_data_agg

,primary_id,time,num_jobs,earning_per_job
0,402402,21.0,30592.907791,163609.620488
1,402403,21.0,7184.221916,69844.53879
2,402404,21.0,10553.138793,114591.865227
3,402405,21.0,10329.819880,61421.909034
4,402406,21.0,27913.284914,240984.76459
...,...,...,...,...
959,403398,21.0,20074.832845,91639.779457
960,403399,21.0,13488.129216,116293.548455
961,403400,21.0,16191.791192,78560.507381
962,403401,21.0,6795.072635,73039.006672


## Merge emissions and cb data with lhs samples

In [192]:
#attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY_6004_filtered_metamodel_version.csv"))
attr_primary_df

,primary_id,design_id,strategy_id,future_id
1,402402,4,6004,0
2,402403,4,6004,1
3,402404,4,6004,2
4,402405,4,6004,3
5,402406,4,6004,4
...,...,...,...,...
997,403398,4,6004,996
998,403399,4,6004,997
999,403400,4,6004,998
1000,403401,4,6004,999


In [193]:
# Check for duplicates in primary_id
duplicates_primary = attr_primary_df[attr_primary_df.duplicated(subset=["primary_id"], keep=False)]
if not duplicates_primary.empty:
    print("Duplicated primary_id found:")
    print(duplicates_primary)
else:
    print("No duplicated primary_id found.")

# Check for duplicates in future_id
duplicates_future = attr_primary_df[attr_primary_df.duplicated(subset=["future_id"], keep=False)]
if not duplicates_future.empty:
    print("Duplicated future_id found:")
    print(duplicates_future)
else:
    print("No duplicated future_id found.")


No duplicated primary_id found.
No duplicated future_id found.


In [194]:
la_emission_df_w_future_id = la_emissions_df_merged.merge(attr_primary_df, on="primary_id", how="inner")

# Drop design_id and stratgy_id columns
la_emission_df_w_future_id = la_emission_df_w_future_id.drop(columns=["design_id", "strategy_id"])
la_emission_df_w_future_id

,primary_id,emission_avg_last_five_years,emission_total,future_id
0,402402,26.097556,3308.232823,0
1,402403,134.250713,4713.726381,1
2,402404,164.387644,5168.255434,2
3,402405,104.151002,4411.805579,3
4,402406,157.680453,5094.995066,4
...,...,...,...,...
959,403398,121.594420,4691.472707,996
960,403399,162.146717,5202.036088,997
961,403400,194.313061,5592.469537,998
962,403401,200.981586,5717.910263,999


In [195]:
la_ssp_out_df = la_emission_df_w_future_id.merge(la_production_df_mean_agg, on="primary_id", how="inner")
la_ssp_out_df = la_ssp_out_df.merge(la_production_cost_df_mean_agg, on="primary_id", how="inner")
la_ssp_out_df.head()

,primary_id,emission_avg_last_five_years,emission_total,future_id,production_total,production_cost_total
0,402402,26.097556,3308.232823,0,251.650034,4.540606e+08
1,402403,134.250713,4713.726381,1,279.612528,4.567414e+08
2,402404,164.387644,5168.255434,2,276.395578,4.708078e+08
3,402405,104.151002,4411.805579,3,264.412432,4.615890e+08
4,402406,157.680453,5094.995066,4,275.458617,4.799190e+08


In [196]:
# Check that the shape is correct
print("la_ssp_out_df shape:", la_ssp_out_df.shape)
print("la_emission_df_w_future_id shape:", la_emission_df_w_future_id.shape)
print("la_production_df_sum_agg shape:", la_production_df_mean_agg.shape)
print("la_production_cost_df_sum_agg shape:", la_production_cost_df_mean_agg.shape)

la_ssp_out_df shape: (964, 6)
la_emission_df_w_future_id shape: (964, 4)
la_production_df_sum_agg shape: (964, 2)
la_production_cost_df_sum_agg shape: (964, 2)


In [197]:
la_ssp_out_df.isna().sum()

primary_id                      0
emission_avg_last_five_years    0
emission_total                  0
future_id                       0
production_total                0
production_cost_total           0
dtype: int64

In [198]:
lhs_df_merged.head()

,future_id,46,47,48,49,50,51,52,53,54,...,1759,1760,1765,1766,1768,1776,1778,1790,1791,1799
2000,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,0.508737,...,0.644272,0.214180,0.955481,0.733736,0.116532,0.828401,0.217715,0.589410,0.939430,0.804396
2001,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,0.973614,...,0.486423,0.278023,0.680104,0.626837,0.392910,0.005622,0.342793,0.767436,0.435921,0.631640
2002,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,0.053175,...,0.331039,0.193177,0.968159,0.467609,0.402445,0.099340,0.676226,0.109182,0.582023,0.877888
2003,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,0.885947,...,0.267909,0.674077,0.246089,0.464519,0.968246,0.733699,0.143231,0.289890,0.794136,0.147144
2004,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,0.223495,...,0.135390,0.111915,0.455861,0.378087,0.267354,0.528790,0.529280,0.660236,0.690165,0.539415


In [199]:
lhs_emissions_merged_df = pd.merge(lhs_df_merged, la_ssp_out_df, on="future_id", how="inner")
lhs_emissions_merged_df.head()

,future_id,46,47,48,49,50,51,52,53,54,...,1776,1778,1790,1791,1799,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total
0,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,0.508737,...,0.828401,0.217715,0.589410,0.939430,0.804396,402403,134.250713,4713.726381,279.612528,4.567414e+08
1,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,0.973614,...,0.005622,0.342793,0.767436,0.435921,0.631640,402404,164.387644,5168.255434,276.395578,4.708078e+08
2,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,0.053175,...,0.099340,0.676226,0.109182,0.582023,0.877888,402405,104.151002,4411.805579,264.412432,4.615890e+08
3,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,0.885947,...,0.733699,0.143231,0.289890,0.794136,0.147144,402406,157.680453,5094.995066,275.458617,4.799190e+08
4,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,0.223495,...,0.528790,0.529280,0.660236,0.690165,0.539415,402407,187.314145,5488.927406,243.699542,4.032488e+08


In [200]:
lhs_emissions_merged_df.shape

(963, 87)

In [201]:
cb_df_agg.head()

,primary_id,future_id,technical_cost,air_pollution
0,402402,0,-8.906431,1.450083
1,402403,1,-2.275659,1.033437
2,402404,2,-7.468213,0.753391
3,402405,3,-3.833560,1.181780
4,402406,4,-10.618699,0.727183


In [202]:
complete_merged_df = pd.merge(lhs_emissions_merged_df, cb_df_agg, on=["future_id", "primary_id"], how="inner")

complete_merged_df.head()

,future_id,46,47,48,49,50,51,52,53,54,...,1790,1791,1799,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution
0,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,0.508737,...,0.589410,0.939430,0.804396,402403,134.250713,4713.726381,279.612528,4.567414e+08,-2.275659,1.033437
1,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,0.973614,...,0.767436,0.435921,0.631640,402404,164.387644,5168.255434,276.395578,4.708078e+08,-7.468213,0.753391
2,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,0.053175,...,0.109182,0.582023,0.877888,402405,104.151002,4411.805579,264.412432,4.615890e+08,-3.833560,1.181780
3,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,0.885947,...,0.289890,0.794136,0.147144,402406,157.680453,5094.995066,275.458617,4.799190e+08,-10.618699,0.727183
4,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,0.223495,...,0.660236,0.690165,0.539415,402407,187.314145,5488.927406,243.699542,4.032488e+08,-6.069652,0.963409


In [203]:
complete_merged_df

,future_id,46,47,48,49,50,51,52,53,54,...,1790,1791,1799,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution
0,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,0.508737,...,0.589410,0.939430,0.804396,402403,134.250713,4713.726381,279.612528,4.567414e+08,-2.275659,1.033437
1,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,0.973614,...,0.767436,0.435921,0.631640,402404,164.387644,5168.255434,276.395578,4.708078e+08,-7.468213,0.753391
2,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,0.053175,...,0.109182,0.582023,0.877888,402405,104.151002,4411.805579,264.412432,4.615890e+08,-3.833560,1.181780
3,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,0.885947,...,0.289890,0.794136,0.147144,402406,157.680453,5094.995066,275.458617,4.799190e+08,-10.618699,0.727183
4,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,0.223495,...,0.660236,0.690165,0.539415,402407,187.314145,5488.927406,243.699542,4.032488e+08,-6.069652,0.963409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
958,996,0.535777,0.643117,0.039013,0.511673,0.770811,0.214445,0.991425,0.880307,0.590308,...,0.160915,0.440383,0.902238,403398,121.594420,4691.472707,248.818823,4.102299e+08,-4.849271,0.827842
959,997,0.776126,0.825906,0.960728,0.400776,0.776082,0.496014,0.795152,0.557662,0.782966,...,0.774330,0.562856,0.730191,403399,162.146717,5202.036088,269.682673,4.655390e+08,-8.026771,0.250140
960,998,0.117222,0.783518,0.845782,0.771561,0.250421,0.733586,0.592633,0.047925,0.482816,...,0.709469,0.210959,0.795728,403400,194.313061,5592.469537,292.994094,5.283043e+08,-5.954081,0.030116
961,999,0.960680,0.188435,0.975613,0.017300,0.272812,0.873458,0.358376,0.094290,0.017529,...,0.990733,0.540858,0.443389,403401,200.981586,5717.910263,282.274217,4.902101e+08,-7.731554,0.563744


In [204]:
complete_merged_df = pd.merge(complete_merged_df, lsu_data_agg, on=[ "primary_id"], how="inner")
complete_merged_df

,future_id,46,47,48,49,50,51,52,53,54,...,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,time,num_jobs,earning_per_job
0,1,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,0.508737,...,402403,134.250713,4713.726381,279.612528,4.567414e+08,-2.275659,1.033437,21.0,7184.221916,69844.53879
1,2,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,0.973614,...,402404,164.387644,5168.255434,276.395578,4.708078e+08,-7.468213,0.753391,21.0,10553.138793,114591.865227
2,3,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,0.053175,...,402405,104.151002,4411.805579,264.412432,4.615890e+08,-3.833560,1.181780,21.0,10329.819880,61421.909034
3,4,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,0.885947,...,402406,157.680453,5094.995066,275.458617,4.799190e+08,-10.618699,0.727183,21.0,27913.284914,240984.76459
4,5,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,0.223495,...,402407,187.314145,5488.927406,243.699542,4.032488e+08,-6.069652,0.963409,21.0,8311.856268,132260.785244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
958,996,0.535777,0.643117,0.039013,0.511673,0.770811,0.214445,0.991425,0.880307,0.590308,...,403398,121.594420,4691.472707,248.818823,4.102299e+08,-4.849271,0.827842,21.0,20074.832845,91639.779457
959,997,0.776126,0.825906,0.960728,0.400776,0.776082,0.496014,0.795152,0.557662,0.782966,...,403399,162.146717,5202.036088,269.682673,4.655390e+08,-8.026771,0.250140,21.0,13488.129216,116293.548455
960,998,0.117222,0.783518,0.845782,0.771561,0.250421,0.733586,0.592633,0.047925,0.482816,...,403400,194.313061,5592.469537,292.994094,5.283043e+08,-5.954081,0.030116,21.0,16191.791192,78560.507381
961,999,0.960680,0.188435,0.975613,0.017300,0.272812,0.873458,0.358376,0.094290,0.017529,...,403401,200.981586,5717.910263,282.274217,4.902101e+08,-7.731554,0.563744,21.0,6795.072635,73039.006672


In [205]:
print(complete_merged_df.shape)
print(complete_merged_df.future_id.nunique())

(963, 92)
963


In [206]:
complete_merged_df.isna().sum().sum()

np.int64(0)

In [207]:
# rearrange columns to have future_id and primary_id at the front
cols_order = ["future_id", "primary_id"] + [col for col in complete_merged_df.columns if col not in ["future_id", "primary_id"]]
complete_merged_df = complete_merged_df[cols_order]
complete_merged_df

,future_id,primary_id,46,47,48,49,50,51,52,53,...,1799,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,time,num_jobs,earning_per_job
0,1,402403,0.336136,0.284572,0.567190,0.653223,0.662849,0.115262,0.601828,0.954157,...,0.804396,134.250713,4713.726381,279.612528,4.567414e+08,-2.275659,1.033437,21.0,7184.221916,69844.53879
1,2,402404,0.862794,0.890697,0.738990,0.685731,0.370917,0.299382,0.979554,0.495744,...,0.631640,164.387644,5168.255434,276.395578,4.708078e+08,-7.468213,0.753391,21.0,10553.138793,114591.865227
2,3,402405,0.462012,0.215267,0.195042,0.179275,0.105911,0.333594,0.088741,0.613166,...,0.877888,104.151002,4411.805579,264.412432,4.615890e+08,-3.833560,1.181780,21.0,10329.819880,61421.909034
3,4,402406,0.188153,0.531601,0.594192,0.559991,0.695983,0.986190,0.743630,0.487156,...,0.147144,157.680453,5094.995066,275.458617,4.799190e+08,-10.618699,0.727183,21.0,27913.284914,240984.76459
4,5,402407,0.708695,0.838746,0.697608,0.889257,0.494082,0.818406,0.375577,0.394310,...,0.539415,187.314145,5488.927406,243.699542,4.032488e+08,-6.069652,0.963409,21.0,8311.856268,132260.785244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
958,996,403398,0.535777,0.643117,0.039013,0.511673,0.770811,0.214445,0.991425,0.880307,...,0.902238,121.594420,4691.472707,248.818823,4.102299e+08,-4.849271,0.827842,21.0,20074.832845,91639.779457
959,997,403399,0.776126,0.825906,0.960728,0.400776,0.776082,0.496014,0.795152,0.557662,...,0.730191,162.146717,5202.036088,269.682673,4.655390e+08,-8.026771,0.250140,21.0,13488.129216,116293.548455
960,998,403400,0.117222,0.783518,0.845782,0.771561,0.250421,0.733586,0.592633,0.047925,...,0.795728,194.313061,5592.469537,292.994094,5.283043e+08,-5.954081,0.030116,21.0,16191.791192,78560.507381
961,999,403401,0.960680,0.188435,0.975613,0.017300,0.272812,0.873458,0.358376,0.094290,...,0.443389,200.981586,5717.910263,282.274217,4.902101e+08,-7.731554,0.563744,21.0,6795.072635,73039.006672


In [208]:
complete_merged_df = complete_merged_df.drop(columns=["time"])

In [209]:
# Check for nans
complete_merged_df.isna().sum().sum()

np.int64(0)

## Filter out irrelevant lhs groups

In [210]:
var_traj_X_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_X.csv"))
var_traj_L_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_L.csv"))

In [211]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
78,elasticity_ippu_wood_production_to_gdp,57
79,elasticity_ippu_product_use_lubricants_product...,58
80,elasticity_ippu_product_use_ods_other_product_...,58
81,elasticity_ippu_product_use_ods_refrigeration_...,58
82,elasticity_ippu_product_use_paraffin_wax_produ...,58


In [212]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
458,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,45
459,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,45
460,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,45
461,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,45
462,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,45


In [213]:
var_traj_groups_X = var_traj_X_df["variable_trajectory_group"].unique()
var_traj_groups_L = var_traj_L_df["variable_trajectory_group"].unique()
print("Variable trajectory groups X:", var_traj_groups_X)
print("Variable trajectory groups L:", var_traj_groups_L)

Variable trajectory groups X: [46 47 48 49 50 51 52 53 54 55 56 57 58]
Variable trajectory groups L: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45]


In [214]:
# join the var_traj_groups
var_traj_groups_all = var_traj_groups_X.tolist() + var_traj_groups_L.tolist()
var_traj_groups_all = list(set(var_traj_groups_all))  # remove duplicates
print("All variable trajectory groups:", var_traj_groups_all)

All variable trajectory groups: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]


In [215]:
# Convert the variable_trajectory_group column to list of strings
relevant_lhs_cols = [str(col) for col in var_traj_groups_all]

In [216]:
df_cols = complete_merged_df.columns.tolist()

# Filter the relevant_lhs_cols to only include those that are in df_cols
relevant_lhs_cols = [col for col in relevant_lhs_cols if col in df_cols]

In [217]:
# filter complete_merged_df to keep only relevant columns
cols_to_keep = ["future_id", "primary_id"] + list(relevant_lhs_cols) + ["emission_avg_last_five_years", "emission_total", "production_total","production_cost_total","technical_cost", "air_pollution", "num_jobs","earning_per_job"]
merged_df_filtered = complete_merged_df[cols_to_keep]

In [218]:
print("Original merged DataFrame shape:", complete_merged_df.shape)
print("Filtered merged DataFrame shape:", merged_df_filtered.shape)

Original merged DataFrame shape: (963, 91)
Filtered merged DataFrame shape: (963, 68)


In [219]:
print("Filtered merged DataFrame fields:", merged_df_filtered.columns.tolist())
print("Relevant LHS columns:", relevant_lhs_cols)

Filtered merged DataFrame fields: ['future_id', 'primary_id', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', 'emission_avg_last_five_years', 'emission_total', 'production_total', 'production_cost_total', 'technical_cost', 'air_pollution', 'num_jobs', 'earning_per_job']
Relevant LHS columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58']


In [220]:
merged_df_filtered.head()

,future_id,primary_id,1,2,3,4,5,6,7,8,...,57,58,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,num_jobs,earning_per_job
0,1,402403,0.073998,0.692822,0.979482,0.051058,0.188261,0.581816,0.698145,0.216501,...,0.792661,0.700647,134.250713,4713.726381,279.612528,4.567414e+08,-2.275659,1.033437,7184.221916,69844.53879
1,2,402404,0.425023,0.729207,0.384489,0.436918,0.621826,0.168596,0.725935,0.167295,...,0.031865,0.682710,164.387644,5168.255434,276.395578,4.708078e+08,-7.468213,0.753391,10553.138793,114591.865227
2,3,402405,0.881846,0.761554,0.325130,0.205597,0.237030,0.682775,0.718216,0.452701,...,0.578509,0.348348,104.151002,4411.805579,264.412432,4.615890e+08,-3.833560,1.181780,10329.819880,61421.909034
3,4,402406,0.075106,0.542313,0.668099,0.577434,0.767079,0.804253,0.966784,0.248884,...,0.536350,0.855946,157.680453,5094.995066,275.458617,4.799190e+08,-10.618699,0.727183,27913.284914,240984.76459
4,5,402407,0.057338,0.477879,0.866591,0.161689,0.282495,0.282664,0.310182,0.991945,...,0.048358,0.042901,187.314145,5488.927406,243.699542,4.032488e+08,-6.069652,0.963409,8311.856268,132260.785244


## Add variable names to lhs columns

In [221]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
458,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,45
459,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,45
460,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,45
461,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,45
462,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,45


In [222]:
var_traj_L_df = var_traj_L_df[["variable_field", "variable_trajectory_group"]]
var_traj_L_df = var_traj_L_df.rename(columns={"variable_field": "variable"})
var_traj_L_df.head()

,variable,variable_trajectory_group
0,ef_agrc_anaerobicdom_rice_kg_ch4_ha,1
1,frac_agrc_agriculture_production_lost,2
2,frac_agrc_crop_residues_burned,3
3,frac_agrc_crop_residues_removed,3
4,frac_agrc_no_till_cereals,3


In [223]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
78,elasticity_ippu_wood_production_to_gdp,57
79,elasticity_ippu_product_use_lubricants_product...,58
80,elasticity_ippu_product_use_ods_other_product_...,58
81,elasticity_ippu_product_use_ods_refrigeration_...,58
82,elasticity_ippu_product_use_paraffin_wax_produ...,58


In [224]:
var_traj_all_df = pd.concat([var_traj_X_df, var_traj_L_df], ignore_index=True)
var_traj_all_df

,variable,variable_trajectory_group
0,cost_enfu_fuel_coal_usd_per_tonne,46
1,cost_enfu_fuel_coke_usd_per_tonne,46
2,cost_enfu_fuel_hydrocarbon_gas_liquids_usd_per...,46
3,cost_enfu_fuel_natural_gas_usd_per_mmbtu,46
4,cost_enfu_fuel_crude_usd_per_m3,46
...,...,...
541,frac_waso_recycled_paper,45
542,frac_waso_recycled_plastic,45
543,frac_waso_recycled_rubber_leather,45
544,frac_waso_recycled_textiles,45


In [225]:
# drop duplicates if any
print("Before dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)
var_traj_all_df = var_traj_all_df.drop_duplicates(subset=["variable", "variable_trajectory_group"])
print("After dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)

Before dropping duplicates, var_traj_all_df shape: (546, 2)
After dropping duplicates, var_traj_all_df shape: (538, 2)


In [226]:
# check if there are any duplicated variable names
duplicated_vars = var_traj_all_df["variable"].duplicated().any()
if duplicated_vars:
    print("There are duplicated variable names in var_traj_all_df.")
else:
    print("No duplicated variable names in var_traj_all_df.")

No duplicated variable names in var_traj_all_df.


In [227]:
# Filter var_traj_all_df by sample_group in relevant_lhs_cols
relevant_lhs_cols = [int(col) for col in relevant_lhs_cols]
var_traj_all_df = var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin(relevant_lhs_cols)]
var_traj_all_df = var_traj_all_df.sort_values(by="variable_trajectory_group", ascending=True)
print("After filtering by relevant_lhs_cols, var_traj_all_df shape:", var_traj_all_df.shape)

After filtering by relevant_lhs_cols, var_traj_all_df shape: (538, 2)


In [228]:
def process_variable_prefix(df):
    result = []
    for group, group_df in df.groupby('variable_trajectory_group'):
        variables = group_df['variable'].tolist()
        if len(variables) == 1:
            prefix = variables[0]
        else:
            prefix = os.path.commonprefix(variables)
            # Clean trailing underscores
            prefix = prefix.rstrip('_')
            
        prefix = f"group_{group}_{prefix}"
        result.append({'variable_trajectory_group': group, 'variable_prefix': prefix})
    return pd.DataFrame(result)

prefix_df = process_variable_prefix(var_traj_all_df)
prefix_df

,variable_trajectory_group,variable_prefix
0,1,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha
1,2,group_2_frac_agrc_agriculture_production_lost
2,3,group_3_frac_agrc
3,4,group_4_qty_ccsq_mt_co2_captured_sequestered_b...
4,5,group_5_nemomod_entc_frac_min_share_production...
5,6,group_6_nemomod_en
6,7,group_7_frac_fgtv_reduction_in_fugitive_leaks
7,8,group_8_frac_fgtv_drained_and_waste_ch4_flared...
8,9,group_9_efficfactor_enfu_industrial_energy_fuel
9,10,group_10_scalar_inen_energy_demand


In [229]:
# Check for duplicates in variable_trajectory_group and variable_prefix
dups = prefix_df.duplicated(subset=["variable_trajectory_group", "variable_prefix"], keep=False)
if dups.any():
    print("Duplicated variable_trajectory_group and variable_prefix found:")
    print(prefix_df[dups])
else:
    print("No duplicated variable_trajectory_group and variable_prefix found.")

# Check for duplicates in variable_trajectory_group
dups_group = prefix_df.duplicated(subset=["variable_trajectory_group"], keep=False)
if dups_group.any():
    print("Duplicated variable_trajectory_group found:")
    print(prefix_df[dups_group])
else:
    print("No duplicated variable_trajectory_group found.")

# Check for duplicates in variable_prefix
dups_prefix = prefix_df.duplicated(subset=["variable_prefix"], keep=False)
if dups_prefix.any():
    print("Duplicated variable_prefix found:")
    print(prefix_df[dups_prefix])
else:
    print("No duplicated variable_prefix found.")

No duplicated variable_trajectory_group and variable_prefix found.
No duplicated variable_trajectory_group found.
No duplicated variable_prefix found.


In [ ]:
# var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin([3, 13, 40])]

In [ ]:
# prefix_df.loc[prefix_df["sample_group"] == 13, "variable_prefix"] = "group_13_frac_gnrl_eating_red_meats+"
# prefix_df.loc[prefix_df["sample_group"] == 40, "variable_prefix"] = "group_40_pij_lndu_grasslands+"

# prefix_df = prefix_df.sort_values(by="variable_prefix", ascending=True)
# prefix_df

In [230]:
# Let's use the prefix_df to rename the columns in merged_df_filtered
def rename_columns_with_prefix(merged_df, prefix_df):
    df = merged_df.copy()
    # Create a mapping from str(group) to prefix
    group_to_prefix = {str(row['variable_trajectory_group']): row['variable_prefix'] for _, row in prefix_df.iterrows()}
    # Only rename columns that match a group
    rename_dict = {col: group_to_prefix[col] for col in df.columns if col in group_to_prefix}
    df = df.rename(columns=rename_dict)
    return df

merged_df_filtered_w_prefix = rename_columns_with_prefix(merged_df_filtered, prefix_df)

In [231]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_2_frac_agrc_agriculture_production_lost,group_3_frac_agrc,group_4_qty_ccsq_mt_co2_captured_sequestered_by_direct_air_capture,group_5_nemomod_entc_frac_min_share_production_fp_hydrogen,group_6_nemomod_en,group_7_frac_fgtv_reduction_in_fugitive_leaks,group_8_frac_fgtv_drained_and_waste_ch4_flared_fuel,...,group_57_elasticity_ippu,group_58_elasticity_ippu_product_use,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,num_jobs,earning_per_job
0,1,402403,0.073998,0.692822,0.979482,0.051058,0.188261,0.581816,0.698145,0.216501,...,0.792661,0.700647,134.250713,4713.726381,279.612528,4.567414e+08,-2.275659,1.033437,7184.221916,69844.53879
1,2,402404,0.425023,0.729207,0.384489,0.436918,0.621826,0.168596,0.725935,0.167295,...,0.031865,0.682710,164.387644,5168.255434,276.395578,4.708078e+08,-7.468213,0.753391,10553.138793,114591.865227
2,3,402405,0.881846,0.761554,0.325130,0.205597,0.237030,0.682775,0.718216,0.452701,...,0.578509,0.348348,104.151002,4411.805579,264.412432,4.615890e+08,-3.833560,1.181780,10329.819880,61421.909034
3,4,402406,0.075106,0.542313,0.668099,0.577434,0.767079,0.804253,0.966784,0.248884,...,0.536350,0.855946,157.680453,5094.995066,275.458617,4.799190e+08,-10.618699,0.727183,27913.284914,240984.76459
4,5,402407,0.057338,0.477879,0.866591,0.161689,0.282495,0.282664,0.310182,0.991945,...,0.048358,0.042901,187.314145,5488.927406,243.699542,4.032488e+08,-6.069652,0.963409,8311.856268,132260.785244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
958,996,403398,0.635142,0.603678,0.650826,0.938293,0.532791,0.458046,0.964056,0.549375,...,0.815408,0.150082,121.594420,4691.472707,248.818823,4.102299e+08,-4.849271,0.827842,20074.832845,91639.779457
959,997,403399,0.523444,0.024740,0.631766,0.447112,0.909923,0.343413,0.604907,0.239868,...,0.710320,0.501973,162.146717,5202.036088,269.682673,4.655390e+08,-8.026771,0.250140,13488.129216,116293.548455
960,998,403400,0.480615,0.560271,0.266827,0.643640,0.040585,0.020424,0.617931,0.743517,...,0.957187,0.471602,194.313061,5592.469537,292.994094,5.283043e+08,-5.954081,0.030116,16191.791192,78560.507381
961,999,403401,0.083011,0.230503,0.294934,0.197481,0.891935,0.140418,0.135728,0.777188,...,0.979377,0.900547,200.981586,5717.910263,282.274217,4.902101e+08,-7.731554,0.563744,6795.072635,73039.006672


In [232]:
print(merged_df_filtered.shape)
print(merged_df_filtered_w_prefix.shape)

(963, 68)
(963, 68)


In [233]:
# check for duplicated column names
duplicated_cols = merged_df_filtered_w_prefix.columns[merged_df_filtered_w_prefix.columns.duplicated()].tolist()
if duplicated_cols:
    print("Duplicated column names found:", duplicated_cols)
else:
    print("No duplicated column names found.")

No duplicated column names found.


## Finally we save the processed data as training data

In [234]:
#save the merged DataFrame to a CSV file
merged_df_filtered_w_prefix.to_csv(os.path.join(TRAINING_DIR_PATH, "training_data_v5.5.csv"), index=False)

In [235]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_2_frac_agrc_agriculture_production_lost,group_3_frac_agrc,group_4_qty_ccsq_mt_co2_captured_sequestered_by_direct_air_capture,group_5_nemomod_entc_frac_min_share_production_fp_hydrogen,group_6_nemomod_en,group_7_frac_fgtv_reduction_in_fugitive_leaks,group_8_frac_fgtv_drained_and_waste_ch4_flared_fuel,...,group_57_elasticity_ippu,group_58_elasticity_ippu_product_use,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,num_jobs,earning_per_job
0,1,402403,0.073998,0.692822,0.979482,0.051058,0.188261,0.581816,0.698145,0.216501,...,0.792661,0.700647,134.250713,4713.726381,279.612528,4.567414e+08,-2.275659,1.033437,7184.221916,69844.53879
1,2,402404,0.425023,0.729207,0.384489,0.436918,0.621826,0.168596,0.725935,0.167295,...,0.031865,0.682710,164.387644,5168.255434,276.395578,4.708078e+08,-7.468213,0.753391,10553.138793,114591.865227
2,3,402405,0.881846,0.761554,0.325130,0.205597,0.237030,0.682775,0.718216,0.452701,...,0.578509,0.348348,104.151002,4411.805579,264.412432,4.615890e+08,-3.833560,1.181780,10329.819880,61421.909034
3,4,402406,0.075106,0.542313,0.668099,0.577434,0.767079,0.804253,0.966784,0.248884,...,0.536350,0.855946,157.680453,5094.995066,275.458617,4.799190e+08,-10.618699,0.727183,27913.284914,240984.76459
4,5,402407,0.057338,0.477879,0.866591,0.161689,0.282495,0.282664,0.310182,0.991945,...,0.048358,0.042901,187.314145,5488.927406,243.699542,4.032488e+08,-6.069652,0.963409,8311.856268,132260.785244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
958,996,403398,0.635142,0.603678,0.650826,0.938293,0.532791,0.458046,0.964056,0.549375,...,0.815408,0.150082,121.594420,4691.472707,248.818823,4.102299e+08,-4.849271,0.827842,20074.832845,91639.779457
959,997,403399,0.523444,0.024740,0.631766,0.447112,0.909923,0.343413,0.604907,0.239868,...,0.710320,0.501973,162.146717,5202.036088,269.682673,4.655390e+08,-8.026771,0.250140,13488.129216,116293.548455
960,998,403400,0.480615,0.560271,0.266827,0.643640,0.040585,0.020424,0.617931,0.743517,...,0.957187,0.471602,194.313061,5592.469537,292.994094,5.283043e+08,-5.954081,0.030116,16191.791192,78560.507381
961,999,403401,0.083011,0.230503,0.294934,0.197481,0.891935,0.140418,0.135728,0.777188,...,0.979377,0.900547,200.981586,5717.910263,282.274217,4.902101e+08,-7.731554,0.563744,6795.072635,73039.006672
